# ForestGuard experiments on Colab (v3)
Runtime > Change runtime type > **T4 GPU**. Needs `fg_code.tar.gz` and `forestguard_data.tar` at the top level of My Drive.
Run cells 1 to 3 in order, then the stages. Every run saves to Drive and is skipped if already finished, so after a disconnect re-run cells 1 to 3 and the stage you were in.

**Frozen training recipe (chosen on validation, seed 0, identical for every model and baseline):** lr 1e-3, output-layer bias init -5, 40 epochs, batch size 16, Adam, cosine schedule, no ImageNet pretraining.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!cp /content/drive/MyDrive/forestguard_data.tar /content/drive/MyDrive/fg_code.tar.gz /content/
!cd /content && tar xf forestguard_data.tar && tar xzf fg_code.tar.gz
%cd /content/fg
!pip -q install affine
import torch, os, pathlib
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change runtime type')
print(len(list(pathlib.Path('/content/raw_tiles').glob('*.npz'))), 'tiles (expect 400)')

In [ ]:
# adds the --prior_bias option to train.py (same change as patch_train.py); safe to re-run
p = '/content/fg/train.py'; s = open(p).read()
if 'prior_bias' in s:
    print('train.py already patched')
else:
    def sub(old, new):
        global s
        assert s.count(old) == 1, f'NOT FOUND exactly once: {old}'
        s = s.replace(old, new)
    sub('ap.add_argument("--max_patches", type=int, default=0)', 'ap.add_argument("--max_patches", type=int, default=0); ap.add_argument("--prior_bias", type=float, default=None)')
    sub('model = build(a.variant, a.pretrained).to(dev)', 'model = build(a.variant, a.pretrained).to(dev)\n    if a.prior_bias is not None: model.head.bias.data.fill_(a.prior_bias)   # start output at the expected positive rate')
    sub('epochs=a.epochs, lr=a.lr, bs=a.bs,', 'epochs=a.epochs, lr=a.lr, bs=a.bs, prior_bias=a.prior_bias,')
    open(p, 'w').write(s); print('patched')

In [ ]:
ROOT = '/content/raw_tiles'
OUT = '/content/drive/MyDrive/forestguard_runs'      # results live on Drive, so they survive a disconnect
EXTRA = '--lr 1e-3 --prior_bias -5'                  # FROZEN recipe, same for every model
os.makedirs(OUT, exist_ok=True)
def run(variant, seed, split='split_seed0.csv', epochs=40, out=OUT):
    d = f"{out}/{variant}_{pathlib.Path(split).stem}_seed{seed}"
    if os.path.exists(f"{d}/result.json"): print('skip (done):', d); return
    print('RUN', variant, split, 'seed', seed, flush=True)
    get_ipython().system(f"python train.py --root {ROOT} --split {split} --variant {variant} --seed {seed} --epochs {epochs} {EXTRA} --out {out} | grep -E '^ep (0|9|19|29|39) |Error|Traceback'")
    assert os.path.exists(f"{d}/result.json"), f"RUN FAILED: {d}  (copy the error above and send it)"
def summary():
    import json, glob
    for f in sorted(glob.glob(f'{OUT}/*/result.json')):
        r = json.load(open(f))
        print(f"{r['variant']:13} {r['split']:28} seed {r['seed']}  best_ep {r['best_epoch']:2d}  val F1 {r['val_F1']:.3f}  test F1 {r['test_pooled']['F1']:.3f}  {r['train_seconds']/60:.1f} min")

## Stage A: main model, 5 seeds (about 45 minutes)

In [ ]:
for s in range(5): run('full', s)
summary()

## Stage B: ablations, 3 seeds each

In [ ]:
for v in ['no_cma', 'early_only', 'late_only', 'optical_only', 'sar_only']:
    for s in range(3): run(v, s)
summary()

## Stage C: baselines, 5 seeds each

In [ ]:
for v in ['unet_r34', 'unet_plain']:
    for s in range(5): run(v, s)
summary()

## Stage D: unseen-region test (train without Congo, test on Congo), 3 seeds

In [ ]:
for v in ['full', 'unet_r34', 'unet_plain']:
    for s in range(3): run(v, s, split='split_seed0_holdout_Congo.csv')
summary()

## Progress

In [ ]:
print(len([d for d in os.listdir(OUT) if os.path.exists(f'{OUT}/{d}/result.json')]), 'finished runs (target 39: 5 + 15 + 10 + 9)')